## Data Extraction with Docling

In this notebook, we'll extract content from PDFs into structured formats:

- **Markdown**: Full document text with page breaks for chunking
- **Images**: Save pages containing large charts/diagrams (>500x500 pixels)
- **Tables**: Extract with 2 paragraphs of context + page number metadata

**Output Structure:**
```
data/rag-data/markdown/{category}/{document}.md
data/rag-data/images/{category}/{document}/page_5.png
data/rag-data/tables/{category}/{document}/table_1_page_5.md
```

https://github.com/docling-project/docling

### 1. Setup and Configuration

In [1]:
from pathlib import Path
from typing import List, Tuple

from docling_core.types.doc import PictureItem
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

In [2]:
# 需要修改
# Directory paths
DATA_DIR = "industrial_data/pdfs"
OUTPUT_MD_DIR = "industrial_data/markdown"
OUTPUT_IMAGES_DIR = "industrial_data/images"
OUTPUT_TABLES_DIR = "industrial_data/tables"

### Metadata Extraction

In [3]:
# 需要修改
def extract_metadata_from_filename(filename: str):
    """
    Extract metadata from filename.

    Expected format: Category_Index_Technology.pdf
    Examples:
        - Industrial_Automation_Safety_Part01_General.pdf
        - Industrial_Automation_Safety_Part02_Pressure_Transmitter.pdf
    """

    filename = filename.replace('.pdf', '').replace('.md', '')
    parts = filename.split("_")

    return {
        'category': "_".join(parts[:3]),
        'index': parts[3],
        'technology': "_".join(parts[4:])
    }

extract_metadata_from_filename("Industrial_Automation_Safety_Part01_General.pdf")

{'category': 'Industrial_Automation_Safety',
 'index': 'Part01',
 'technology': 'General'}

### Extract Markdown

In [4]:
def convert_pdf_to_docling(pdf_file: Path):

    pipeline_options = PdfPipelineOptions()
    pipeline_options.images_scale = 2
    pipeline_options.generate_picture_images = True
    pipeline_options.generate_page_images = True

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    return doc_converter.convert(pdf_file)

In [5]:
def save_page_images(doc_converter, images_dir: Path):
    """
    Find and save pages with large images (>500x500 pixels).
    """

    pages_to_save = set()

    for item in doc_converter.document.iterate_items():
        element = item[0]

        if isinstance(element, PictureItem):
            image = element.get_image(doc_converter.document)

            if image.size[0]>500 and image.size[1]>500:
                page_no = element.prov[0].page_no if element.prov else None

                if page_no:
                    pages_to_save.add(page_no)


        # save images
        for page_no in pages_to_save:
            page = doc_converter.document.pages[page_no]

            page.image.pil_image.save(images_dir/ f"page_{page_no}.png", "PNG")


In [6]:
def extract_context_and_table(lines: List[str], table_index: int):
    """
    Extract context and table content at a specific position.
    
    Args:
        lines: All markdown lines
        table_index: Where the table starts
    
    Returns:
        (combined_content, next_line_index)
    """

    table_lines = []
    i = table_index

    while (i < len(lines)) and (lines[i].startswith('|')):
        table_lines.append(lines[i])
        i = i + 1

    # 可根据需求修改
    # previous 3 lines as table context
    start = max(0, table_index-3)
    context_lines = lines[start: table_index]

    content = '\n'.join(context_lines) + '\n\n' + '\n'.join(table_lines)

    return content, i
    

In [7]:
def extract_tables_with_context(markdown_text: str):
    """
    Find all tables and extract them with context and page numbers.
    
    Returns:
        List of (content, table_name, page_number)
    """

    lines = markdown_text.split('\n')
    lines = [line for line in lines if line.strip()]
    tables = []
    current_page = 1
    table_num = 1
    i = 0

    while(i< len(lines)):
        # track page numbers
        if '<!-- page break -->' in lines[i]:
            current_page = current_page + 1
            i = i + 1
            continue

        # Table detected
        if lines[i].startswith('|') and lines[i].count('|')>1:
            content, next_i = extract_context_and_table(lines, i)

            tables.append((content, f"table_{table_num}", current_page))
            table_num = table_num + 1
            i = next_i

        else:
            i = i + 1


    return tables
    

In [8]:
def save_tables(markdown_text, tables_dir):

    tables = extract_tables_with_context(markdown_text)

    for table_content, table_name, page_num in tables:
        content_with_page = f"**Page:** {page_num}\n\n{table_content}"
                
        (tables_dir/f"{table_name}_page_{page_num}.md").write_text(content_with_page, encoding='utf-8')


In [9]:
# 需要修改
def extract_pdf_content(pdf_file):
    metadata = extract_metadata_from_filename(pdf_file.stem)

    md_dir = Path(OUTPUT_MD_DIR)
    images_dir = Path(OUTPUT_IMAGES_DIR) / pdf_file.stem
    tables_dir = Path(OUTPUT_TABLES_DIR) / pdf_file.stem

    for dir_path in [md_dir, images_dir, tables_dir]:
        dir_path.mkdir(parents=True, exist_ok=True)


    doc_converter = convert_pdf_to_docling(pdf_file)

    markdown_text = doc_converter.document.export_to_markdown(page_break_placeholder="<!-- page break -->")

    (md_dir / f"{pdf_file.stem}.md").write_text(markdown_text, encoding='utf-8')

    save_page_images(doc_converter, images_dir)

    save_tables(markdown_text, tables_dir)


In [18]:
# pdf_file = Path('data\\rag-data\\pdfs\\apple\\apple 8-k q4 2023.pdf')

# extract_pdf_content(pdf_file)

data_path = Path(DATA_DIR)
data_path


WindowsPath('industrial_data/pdfs')

In [19]:
pdf_files = list(data_path.rglob("*.pdf"))
pdf_files

[WindowsPath('industrial_data/pdfs/Industrial_Automation_Safety_Part01_General.pdf'),
 WindowsPath('industrial_data/pdfs/Industrial_Automation_Safety_Part02_Pressure_Transmitter.pdf'),
 WindowsPath('industrial_data/pdfs/Industrial_Automation_Safety_Part03_Temperature_Transmitter.pdf'),
 WindowsPath('industrial_data/pdfs/Industrial_Automation_Safety_Part04_Control_Valve.pdf'),
 WindowsPath('industrial_data/pdfs/Industrial_Automation_Safety_Part05_Flowmeter.pdf'),
 WindowsPath('industrial_data/pdfs/Industrial_Automation_Safety_Part06_Solenoid_Valve.pdf'),
 WindowsPath('industrial_data/pdfs/Industrial_Automation_Safety_Part07_Loop_Regulator.pdf'),
 WindowsPath('industrial_data/pdfs/Industrial_Automation_Safety_Part08_Electric_Actuator.pdf'),
 WindowsPath('industrial_data/pdfs/Industrial_Automation_Safety_Part09_Digital_Display.pdf'),
 WindowsPath('industrial_data/pdfs/Industrial_Automation_Safety_Part10_Recording_Instrument.pdf')]

In [ ]:
for idx, pdf_file in enumerate(pdf_files):
    print(pdf_file)
    extract_pdf_content(pdf_file)